In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *
import pprint
init_demo()
# Reload the global vars again
from resources.utils import *  

from resources.dask_clusters.dask_main_env import *
await init_dask_cluster_staging()

pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_CORES_EOPF=4 docker compose up # ...
#display(dask_cluster_staging)  ==> Because it runs in a separate process we can't display staging clusters anymore

# Search by externalIds

The items of the stations (PRIP, AUXIP, CADIP) carry the identifier of the product in the station in their
`externalIds` property: `[{"scheme": "prip", "value": "<OData Id>"}]`. A searched identifier is:

- `scheme:value`: both the scheme and the value of an external identifier of the item match,
- `value` (or `:value`): only the value matches,
- `scheme:`: only the scheme matches.

Each search below checks the exact items that it finds, so the notebook fails if the search is broken.

In [ ]:
# Products of the station mockups used by this notebook
PRIP_DS = ("prip", "4db05e5e-16d7-4c15-8ca1-9a7d31d06eba")  # S2B_OPER_MSI_L0__DS_2BPS_20250801T074015...
PRIP_GR = ("prip", "b99c8f80-ee84-4854-bd36-15b18ac0ecca")  # S2B_OPER_MSI_L0__GR_2BPS_20250801T074015...
AUXIP_OBMEMC = ("auxip", "99499477-0deb-4b5f-92bd-413e4136ba48")  # S1A_OPER_AUX_OBMEMC_PDMC_20210123T000000.xml
AUXIP_ORBPRE = ("auxip", "0a88d2a2-6e73-41a1-abce-2cdf29dfcf04")  # S1A_OPER_MPL_ORBPRE_20210214T021411...
CADIP_2020 = ("cadip", "a5e9d3b8-7c4f-4a92-b76a-fa09e1e1b59c")  # session S1A_20200105072204051312
CADIP_2021 = ("cadip", "6f3c8d91-2b0e-492d-aef6-87b24f2bcb1e")  # session S1A_20210410031928012345


def check_found(items, *expected):
    """Check that the search found exactly one item for each expected (scheme, value) external identifier."""
    items = list(items or [])
    found = sorted(
        (external_id["scheme"], external_id["value"])
        for item in items
        for external_id in item.properties.get("externalIds", [])
    )
    pp.pprint([item.id for item in items])
    assert len(items) == len(expected), f"{len(items)} items found instead of {len(expected)}: {found}"
    assert found == sorted(expected), f"Found externalIds {found} instead of {sorted(expected)}"
    return items

## Search in the stations

In [ ]:
item_collection_prip = prip_client.search(
    method='GET',
    stac_filter="externalIds='prip:4db05e5e-16d7-4c15-8ca1-9a7d31d06eba, b99c8f80-ee84-4854-bd36-15b18ac0ecca'")
check_found(item_collection_prip, PRIP_DS, PRIP_GR)

In [ ]:
stac_filter = {
  "op": "=",
  "args": [
    { "property": "externalIds" },
    "4db05e5e-16d7-4c15-8ca1-9a7d31d06eba, prip:b99c8f80-ee84-4854-bd36-15b18ac0ecca"
  ]
}
check_found(prip_client.search(method='POST', stac_filter = stac_filter, collections = []), PRIP_DS, PRIP_GR)

In [ ]:
item_collection_auxip = auxip_client.search(
    method="GET",
    stac_filter="externalIds='auxip:99499477-0deb-4b5f-92bd-413e4136ba48, 0a88d2a2-6e73-41a1-abce-2cdf29dfcf04'")
check_found(item_collection_auxip, AUXIP_OBMEMC, AUXIP_ORBPRE)

In [ ]:
stac_filter = {
  "op": "=",
  "args": [
    { "property": "externalIds" },
    "99499477-0deb-4b5f-92bd-413e4136ba48, auxip:0a88d2a2-6e73-41a1-abce-2cdf29dfcf04"
  ]
}
check_found(auxip_client.search(method="POST", stac_filter = stac_filter, collections = []), AUXIP_OBMEMC, AUXIP_ORBPRE)

In [ ]:
item_collection_cadip = cadip_client.search(
    method="GET", 
    stac_filter="externalIds='a5e9d3b8-7c4f-4a92-b76a-fa09e1e1b59c, cadip:6f3c8d91-2b0e-492d-aef6-87b24f2bcb1e'")
check_found(item_collection_cadip, CADIP_2020, CADIP_2021)

In [ ]:
stac_filter = {
  "op": "=",
  "args": [
    { "property": "externalIds" },
    "a5e9d3b8-7c4f-4a92-b76a-fa09e1e1b59c, cadip:6f3c8d91-2b0e-492d-aef6-87b24f2bcb1e"
  ]
}
check_found(cadip_client.search(method="POST", stac_filter = stac_filter, collections = []), CADIP_2020, CADIP_2021)

## Stage the found items into a test collection of the catalog

In [ ]:
# Create a test collection 
CATALOG_COLLECTION_ID = "SPRINT33_STORY906_TEST_COLLECTION"
collection = create_test_collection(CATALOG_COLLECTION_ID)
items = catalog_client.get_items(CATALOG_COLLECTION_ID)
list(items)

In [ ]:
from pystac import ItemCollection

items = (
    list(item_collection_auxip)
    + list(item_collection_cadip)
    + list(item_collection_prip)
)

merged = ItemCollection(items)

In [ ]:
stage_data(merged.to_dict(), items_id = [], catalog_collection_name = CATALOG_COLLECTION_ID)

## Search in the catalog

The searches are restricted to the test collection, because other notebooks may stage the same products in other
collections.

In [ ]:
# Value only (auxip), scheme and value (cadip, prip)
check_found(
    catalog_client.search(
        method="GET",
        collections=[CATALOG_COLLECTION_ID],
        stac_filter="externalIds = '0a88d2a2-6e73-41a1-abce-2cdf29dfcf04, cadip:6f3c8d91-2b0e-492d-aef6-87b24f2bcb1e, prip:b99c8f80-ee84-4854-bd36-15b18ac0ecca'",
        filter_lang="cql2-text",
    ),
    AUXIP_ORBPRE, CADIP_2021, PRIP_GR,
)

In [ ]:
stac_filter = {
  "op": "=",
  "args": [
    { "property": "externalIds" },
    "auxip:0a88d2a2-6e73-41a1-abce-2cdf29dfcf04, cadip:6f3c8d91-2b0e-492d-aef6-87b24f2bcb1e, b99c8f80-ee84-4854-bd36-15b18ac0ecca"
  ]
}

check_found(
    catalog_client.search(method="POST", collections=[CATALOG_COLLECTION_ID], stac_filter=stac_filter),
    AUXIP_ORBPRE, CADIP_2021, PRIP_GR,
)

In [ ]:
# Scheme only: all the items with an external identifier of this scheme, whatever its value
for method in ("GET", "POST"):
    check_found(
        catalog_client.search(
            method=method, collections=[CATALOG_COLLECTION_ID], stac_filter="externalIds = 'prip:'", filter_lang="cql2-text",
        ),
        PRIP_DS, PRIP_GR,
    )

In [ ]:
# Scheme and value: the value of another scheme doesn't match
check_found(
    catalog_client.search(
        method="GET",
        collections=[CATALOG_COLLECTION_ID],
        stac_filter="externalIds = 'auxip:b99c8f80-ee84-4854-bd36-15b18ac0ecca'",
        filter_lang="cql2-text",
    ),
)

In [ ]:
# Several conditions: an identifier with any scheme among the auxip items, and all of a scheme and a value
check_found(
    catalog_client.search(
        method="GET",
        collections=[CATALOG_COLLECTION_ID],
        stac_filter="externalIds IN ('cadip:', '99499477-0deb-4b5f-92bd-413e4136ba48') AND NOT (externalIds = 'cadip:a5e9d3b8-7c4f-4a92-b76a-fa09e1e1b59c')",
        filter_lang="cql2-text",
    ),
    AUXIP_OBMEMC, CADIP_2021,
)
check_found(
    catalog_client.search(
        method="GET",
        collections=[CATALOG_COLLECTION_ID],
        stac_filter="A_CONTAINS(externalIds, ('prip:', '4db05e5e-16d7-4c15-8ca1-9a7d31d06eba'))",
        filter_lang="cql2-text",
    ),
    PRIP_DS,
)

## Clean up

In [ ]:
result = catalog_client.remove_collection(CATALOG_COLLECTION_ID)
assert result.json()["deleted collection"] == CATALOG_COLLECTION_ID
pp.pprint(result.json())